# Predicting startup exits from the first funding round

**MGIS 355 · Project 2 · Aazan Sohail**

**Decision.** A VC associate or accelerator screener decides, one newly funded startup at a time, whether to take a first meeting.
**Grain.** One row per company, as of the date of its first recorded funding round.
**Target.** `exited = 1` if the company was acquired or went public within 5 years of its first funding round, else 0.

The notebook follows the five steps from class: **load, check, split, fit, evaluate**. It runs top to bottom from the repo root
and reads only the CSVs in `data/`. Every random step uses a fixed seed, so two runs give the same numbers.

Setup: imports and every constant the analysis depends on, in one place so a reader can see the assumptions.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

DATA = Path("data")
SEED = 42
SNAPSHOT_END = pd.Timestamp("2013-12-12")       # last date in the Crunchbase 2013 snapshot
WINDOW_YEARS = 5                                 # exit must happen within 5 years of the first round
LAST_FIRST_ROUND = SNAPSHOT_END - pd.DateOffset(years=WINDOW_YEARS)   # 2008-12-12
EARLIEST_FIRST_ROUND = pd.Timestamp("2000-01-01")  # see Problem 1 below

# Cost of each mistake, in "wasted-meeting" units (the decision only depends on the ratio)
COST_FP = 1      # take a meeting with a startup that does not exit
COST_FN = 10     # pass on a startup that exits within 5 years
MEETING_COST_USD = 300   # assumption: about 3 hours of associate time (prep, meeting, notes) at ~$100/hour
print("Prediction window: first rounds from", EARLIEST_FIRST_ROUND.date(), "to", LAST_FIRST_ROUND.date())

Prediction window: first rounds from 2000-01-01 to 2008-12-12


## Step 1 · Load

Load the four tables trimmed from the Kaggle download by `data/make_extracts.py` (companies only; no names, URLs or people).

In [2]:
companies = pd.read_csv(DATA / "companies.csv", low_memory=False)
rounds = pd.read_csv(DATA / "funding_rounds.csv")
acq = pd.read_csv(DATA / "acquisitions.csv")
ipos = pd.read_csv(DATA / "ipos.csv")
for name, t in [("companies", companies), ("funding_rounds", rounds), ("acquisitions", acq), ("ipos", ipos)]:
    print(f"{name:15s} {len(t):>8,} rows  {t.shape[1]} columns")

companies        196,553 rows  8 columns
funding_rounds    52,928 rows  8 columns
acquisitions       9,562 rows  3 columns
ipos               1,259 rows  3 columns


## Step 2 · Check

Parse every date column, and count the dates that are missing or unreadable, because the target and the split both depend on dates.

In [3]:
rounds["funded_at"] = pd.to_datetime(rounds["funded_at"], errors="coerce")
rounds["created_at"] = pd.to_datetime(rounds["created_at"], errors="coerce")
acq["acquired_at"] = pd.to_datetime(acq["acquired_at"], errors="coerce")
ipos["public_at"] = pd.to_datetime(ipos["public_at"], errors="coerce")
companies["founded_at"] = pd.to_datetime(companies["founded_at"], errors="coerce")

print("funding rounds with no date:", rounds["funded_at"].isna().sum())
print("funding rounds on objects that are not companies (investors, people):", (~rounds["object_id"].isin(companies["id"])).sum())
print("acquisitions with no date:", acq["acquired_at"].isna().sum(), "of", len(acq))
print("IPOs with no date:", ipos["public_at"].isna().sum(), "of", len(ipos))

funding rounds with no date: 248


funding rounds on objects that are not companies (investors, people): 302
acquisitions with no date: 29 of 9562
IPOs with no date: 600 of 1259


Keep only dated funding rounds that belong to companies, and find each company's first round date.

In [4]:
r = rounds[rounds["object_id"].isin(companies["id"]) & rounds["funded_at"].notna()].copy()
r["first_date"] = r.groupby("object_id")["funded_at"].transform("min")
first_date = r.groupby("object_id")["funded_at"].min()
print("companies with at least one dated funding round:", f"{len(first_date):,}")

companies with at least one dated funding round: 31,507


**Problem 1 · placeholder dates.** Some first-round and founding dates are filler values, not real dates. Count them.

In [5]:
print("first rounds dated before 2000:", (first_date < "2000-01-01").sum())
print("  of which exactly 1960-01-01:", (first_date == "1960-01-01").sum())
print("founding dates exactly 1901-01-01:", (companies["founded_at"] == "1901-01-01").sum())
print("share of founding dates on January 1 (year-only precision):",
      round((companies["founded_at"].dt.strftime("%m-%d") == "01-01").mean(), 3))

first rounds dated before 2000: 120
  of which exactly 1960-01-01: 7
founding dates exactly 1901-01-01: 5
share of founding dates on January 1 (year-only precision): 0.214


**Problem 2 · zeros that mean missing.** A round with $0 raised or 0 investors is a round where Crunchbase did not record the value.

In [6]:
first_rounds = r[r["funded_at"] == r["first_date"]]
print("first rounds with raised_amount_usd == 0:", round((first_rounds["raised_amount_usd"] == 0).mean(), 3))
print("first rounds with participants == 0:     ", round((first_rounds["participants"] == 0).mean(), 3))

first rounds with raised_amount_usd == 0: 0.146
first rounds with participants == 0:      0.37


**Problem 3 · exits with no date.** Half the IPO rows have no `public_at`, and some companies say `acquired` or `ipo` with no dated exit at all, so we cannot tell whether the exit fell inside the 5-year window.

In [7]:
exit_dates = pd.concat([
    acq.groupby("acquired_object_id")["acquired_at"].min().rename("acq_at"),
    ipos.groupby("object_id")["public_at"].min().rename("ipo_at"),
], axis=1)
exit_dates["exit_at"] = exit_dates.min(axis=1)
undated_ids = set(acq.loc[acq["acquired_at"].isna(), "acquired_object_id"]) | set(ipos.loc[ipos["public_at"].isna(), "object_id"])
print("IPO rows with no date:", ipos["public_at"].isna().sum(), "of", len(ipos))
print("companies listed in acquisitions/ipos with an undated row:", len(undated_ids))

IPO rows with no date: 600 of 1259
companies listed in acquisitions/ipos with an undated row: 624


**Problem 4 · one round type coded two ways.** `funding_round_type` and `funding_round_code` disagree: many 'series-a' rows are coded seed or angel. Cross-tab them.

In [8]:
print(pd.crosstab(rounds["funding_round_type"], rounds["funding_round_code"]).loc[["angel", "series-a", "venture", "other"]].loc[:, lambda t: t.sum() > 0])

funding_round_code     a  angel  convertible  crowd_equity  debt_round  grant  partial  secondary_market   seed  unattributed
funding_round_type                                                                                                           
angel                  0   2826          146             2           0      0        0                 0  10189             0
series-a            7999    413           41             1           0      0        0                 0   1419             0
venture                0      0            0             0           0      0     3115                 0      0         12227
other                  0      0            0             0        3409    776        0                16      0             0


Map type and code to one `stage` label, letting the code win when it says seed or angel. This is the fix for Problem 4.

In [9]:
STAGE_ORDER = ["series-b-or-later", "series-a", "venture-unlabeled", "seed-angel", "debt", "grant", "other"]

def round_stage(rtype, rcode):
    if rcode in ("seed", "angel"):
        return "seed-angel"
    if rtype == "series-a":
        return "series-a"
    if rtype in ("series-b", "series-c+"):
        return "series-b-or-later"
    if rtype == "venture":
        return "venture-unlabeled"
    if rtype == "angel":
        return "seed-angel"
    if rcode == "debt_round":
        return "debt"
    if rcode == "grant":
        return "grant"
    return "other"

first_rounds = first_rounds.assign(stage=[round_stage(t, c) for t, c in zip(first_rounds["funding_round_type"], first_rounds["funding_round_code"])])
first_rounds["stage_rank"] = first_rounds["stage"].map({s: i for i, s in enumerate(STAGE_ORDER)})
print(first_rounds["stage"].value_counts())

stage
seed-angel           11695
venture-unlabeled     9226
series-a              5584
series-b-or-later     2684
debt                  1411
other                  684
grant                  480
Name: count, dtype: int64


**Problem 5 · duplicates.** Some companies have several acquisition rows or several rounds on their first date. Count them; the fix is to take the earliest exit and to combine same-day rounds.

In [10]:
print("companies with more than one acquisition row:", acq["acquired_object_id"].duplicated().sum())
print("companies with more than one IPO row:", ipos["object_id"].duplicated().sum())
print("companies with 2+ rounds on their first date:", first_rounds["object_id"].duplicated().sum())

companies with more than one acquisition row: 150
companies with more than one IPO row: 5
companies with 2+ rounds on their first date: 257


Collapse to one row per company at its first round: sum amount and investors across same-day rounds, keep the most advanced stage.

In [11]:
fr1 = first_rounds.sort_values(["object_id", "stage_rank", "funding_round_id"])
co = fr1.groupby("object_id").agg(
    first_date=("funded_at", "min"),
    stage=("stage", "first"),
    amount_raw=("raised_amount_usd", "sum"),
    investors_raw=("participants", "sum"),
    record_created=("created_at", "min"),
)
co = co.join(companies.set_index("id")[["category_code", "status", "founded_at", "country_code", "state_code",
                                        "funding_rounds", "funding_total_usd"]])
co = co.join(exit_dates["exit_at"])
print(len(co), "companies, one row each")

31507 companies, one row each


Apply the target's exclusions in order and print the funnel. The first rows reproduce the checkpoint; the last two are new fixes (Problems 1 and 3).

In [12]:
funnel = [("Companies with at least one dated funding round", len(co))]
co = co[~(co["exit_at"] < co["first_date"])]
funnel.append(("After dropping exits dated before the first round", len(co)))
co = co[co["first_date"] <= LAST_FIRST_ROUND]
funnel.append(("First round on or before 2008-12-12 (full 5-year window) [checkpoint rows]", len(co)))
co = co[co["first_date"] >= EARLIEST_FIRST_ROUND]
funnel.append(("First round on or after 2000-01-01 (drops placeholder/backfilled dates)", len(co)))
undated_exit = co["exit_at"].isna() & (co["status"].isin(["acquired", "ipo"]) | co.index.isin(undated_ids))
co = co[~undated_exit]
funnel.append(("After dropping companies that exited on an unknown date", len(co)))

co["exited"] = (co["exit_at"].notna() & (co["exit_at"] <= co["first_date"] + pd.DateOffset(years=WINDOW_YEARS))).astype(int)
print(pd.DataFrame(funnel, columns=["step", "rows"]).to_string(index=False))
print(f"\nModeling rows: {len(co):,}   exited within 5 years: {co['exited'].sum():,}   base rate: {co['exited'].mean():.1%}")

                                                                      step  rows
                           Companies with at least one dated funding round 31507
                         After dropping exits dated before the first round 31305
First round on or before 2008-12-12 (full 5-year window) [checkpoint rows]  8369
   First round on or after 2000-01-01 (drops placeholder/backfilled dates)  8249
                   After dropping companies that exited on an unknown date  8214

Modeling rows: 8,214   exited within 5 years: 1,185   base rate: 14.4%


**Problem 6 · back-filled records.** Crunchbase started in 2007, so older rounds were typed in years later, often after the company had already exited. Measure the lag between a round and the day its record was created.

In [13]:
lag = (co["record_created"] - co["first_date"]).dt.days
print("earliest record created:", rounds["created_at"].min().date())
print("median days from first round to record creation, by first-round year:")
print(lag.groupby(co["first_date"].dt.year).median().astype(int).to_string())
pos = co[co["exited"] == 1]
print("exits whose first-round record was created after the exit itself:", f"{(pos['record_created'] > pos['exit_at']).mean():.1%}")

earliest record created: 2007-05-27
median days from first round to record creation, by first-round year:
first_date
2000    3676
2001    3216
2002    2965
2003    2392
2004    1830
2005    1866
2006    1375
2007     806
2008     342
exits whose first-round record was created after the exit itself: 45.5%


Build the features, using only information dated on or before the first round. Zeros become missing (Problem 2) and impossible dates become missing (Problem 1).

In [14]:
co["amount_usd"] = co["amount_raw"].where(co["amount_raw"] > 0)
co["n_investors"] = co["investors_raw"].where(co["investors_raw"] > 0)

founded = co["founded_at"].where(co["founded_at"] >= "1980-01-01")
age = (co["first_date"] - founded).dt.days / 365.25
co["company_age_years"] = age.where(age >= -1).clip(lower=0)   # founded >1 year after its own first round = bad date
print("first round dated before the founding date:", int((age < 0).sum()), " of which by more than a year (set to missing):", int((age < -1).sum()))

TOP_COUNTRIES = ["USA", "GBR", "CAN", "ISR", "DEU", "FRA", "IND"]
co["country"] = np.where(co["country_code"].isna(), "unknown",
                         np.where(co["country_code"].isin(TOP_COUNTRIES), co["country_code"], "other"))
us_state = co["state_code"].where(co["country_code"] == "USA")
co["us_state"] = np.where(co["country_code"] != "USA", "non-US",
                          np.where(us_state.isna(), "US-unknown",
                                   np.where(us_state.isin(["CA", "NY", "MA", "WA", "TX"]), us_state, "US-other")))
co["category"] = co["category_code"].fillna("unknown")

NUM = ["amount_usd", "n_investors", "company_age_years"]
CAT = ["stage", "category", "country", "us_state"]
FEATURES = NUM + CAT
print(co[NUM].describe().round(2))
print("\nmissing share:", co[NUM].isna().mean().round(3).to_dict())

first round dated before the founding date: 594  of which by more than a year (set to missing): 171


         amount_usd  n_investors  company_age_years
count  7.260000e+03      6883.00            6261.00
mean   8.537105e+06         2.24               3.55
std    2.194611e+07         1.73               4.13
min    1.000000e+03         1.00               0.00
25%    1.000000e+06         1.00               0.57
50%    4.000000e+06         2.00               2.12
75%    9.150000e+06         3.00               5.36
max    1.000000e+09        23.00              28.67

missing share: {'amount_usd': 0.116, 'n_investors': 0.162, 'company_age_years': 0.238}


The time test, written down: every feature, and whether it exists on the day of the first round.

In [15]:
time_test = pd.DataFrame([
    ("amount_usd", "First-round amount in USD (sum of same-day rounds)", "yes"),
    ("n_investors", "Investors in the first round", "yes"),
    ("company_age_years", "Years from founding to first round", "yes"),
    ("stage", "Round stage (from type + code)", "yes"),
    ("category", "Industry category", "yes (set at profile creation; could be edited later)"),
    ("country / us_state", "Headquarters location", "yes (could be edited later)"),
    ("first-round year", "Calendar year of the first round", "yes, but EXCLUDED: every new case is outside the training years"),
    ("funding_rounds, funding_total_usd", "Round count and total raised as of 2013", "NO: counts later rounds"),
    ("status", "operating / acquired / ipo / closed as of 2013", "NO: this is the outcome"),
    ("id, name, URLs", "Identifiers", "excluded: not features"),
], columns=["feature", "meaning", "known at prediction time?"])
print(time_test.to_string(index=False))

                          feature                                            meaning                                       known at prediction time?
                       amount_usd First-round amount in USD (sum of same-day rounds)                                                             yes
                      n_investors                       Investors in the first round                                                             yes
                company_age_years                 Years from founding to first round                                                             yes
                            stage                     Round stage (from type + code)                                                             yes
                         category                                  Industry category            yes (set at profile creation; could be edited later)
               country / us_state                              Headquarters location                      

## Step 3 · Split

Time split: sort by first-round date, train on the earliest ~80%, test on the latest ~20%. All companies funded on the cutoff date go to test, so no date sits on both sides.

In [16]:
co = co.reset_index().rename(columns={"index": "object_id"}).sort_values(["first_date", "object_id"]).reset_index(drop=True)
TEST_START = co["first_date"].iloc[int(len(co) * 0.8)]
train = co[co["first_date"] < TEST_START].copy()
test = co[co["first_date"] >= TEST_START].copy()

VAL_START = train["first_date"].iloc[int(len(train) * 0.8)]
fit_part = train[train["first_date"] < VAL_START]      # used to fit models while tuning
val = train[train["first_date"] >= VAL_START]          # used to pick settings and the threshold (never test)

split_tbl = pd.DataFrame({
    "rows": [len(fit_part), len(val), len(train), len(test)],
    "from": [fit_part.first_date.min(), val.first_date.min(), train.first_date.min(), test.first_date.min()],
    "to": [fit_part.first_date.max(), val.first_date.max(), train.first_date.max(), test.first_date.max()],
    "exits": [fit_part.exited.sum(), val.exited.sum(), train.exited.sum(), test.exited.sum()],
    "base_rate": [fit_part.exited.mean(), val.exited.mean(), train.exited.mean(), test.exited.mean()],
}, index=["fit (train part)", "validation (train part)", "train (all)", "test"])
split_tbl["from"] = split_tbl["from"].dt.date
split_tbl["to"] = split_tbl["to"].dt.date
print(split_tbl.round(3))

                         rows        from          to  exits  base_rate
fit (train part)         5252  2000-01-01  2007-08-14    834      0.159
validation (train part)  1314  2007-08-15  2008-02-24    162      0.123
train (all)              6566  2000-01-01  2008-02-24    996      0.152
test                     1648  2008-02-25  2008-12-12    189      0.115


## Step 4 · Fit

Helper functions for the cost of a set of meeting decisions, written once and used for every model and baseline.

In [17]:
def confusion(y, flag):
    y = np.asarray(y).astype(int)
    flag = np.asarray(flag).astype(bool)
    return dict(tp=int(((flag) & (y == 1)).sum()), fp=int(((flag) & (y == 0)).sum()),
                fn=int(((~flag) & (y == 1)).sum()), tn=int(((~flag) & (y == 0)).sum()))

def cost(y, flag, c_fp=COST_FP, c_fn=COST_FN):
    c = confusion(y, flag)
    return c["fp"] * c_fp + c["fn"] * c_fn

THRESHOLDS = np.round(np.arange(0.01, 0.601, 0.005), 3)

def best_threshold(y, p, c_fp=COST_FP, c_fn=COST_FN):
    costs = [cost(y, p >= t, c_fp, c_fn) for t in THRESHOLDS]
    i = int(np.argmin(costs))
    return float(THRESHOLDS[i]), int(costs[i])

def scorecard(name, y, flag, p=None):
    c = confusion(y, flag)
    n = len(y)
    row = dict(rule=name, meetings_per_100=100 * (c["tp"] + c["fp"]) / n, exits_caught=c["tp"], exits_missed=c["fn"],
               recall=c["tp"] / max(c["tp"] + c["fn"], 1), precision=c["tp"] / max(c["tp"] + c["fp"], 1),
               accuracy=(c["tp"] + c["tn"]) / n, cost_units=c["fp"] * COST_FP + c["fn"] * COST_FN)
    row["cost_per_100_usd"] = row["cost_units"] / n * 100 * MEETING_COST_USD
    row["auc"] = roc_auc_score(y, p) if p is not None else np.nan
    return row

print("theory: with calibrated probabilities, the cost-minimizing threshold is", round(COST_FP / (COST_FP + COST_FN), 3))

theory: with calibrated probabilities, the cost-minimizing threshold is 0.091


Baseline 0 is 'predict no exit for everyone' (pass on every startup). It shows why accuracy is the wrong yardstick on a rare target.

In [18]:
b0_train = np.zeros(len(train), bool)
print(pd.DataFrame([scorecard("predict no exit", train["exited"], b0_train)]).round(3).to_string(index=False))

           rule  meetings_per_100  exits_caught  exits_missed  recall  precision  accuracy  cost_units  cost_per_100_usd  auc
predict no exit               0.0             0           996     0.0        0.0     0.848        9960         45507.158  NaN


Baseline 1 is a rule a screener might already use: meet if the first round is Series A or later, or raised at least $X. Pick $X on the training set by cost.

In [19]:
LATER_STAGE = ["series-a", "series-b-or-later"]

def baseline_rule(d, cutoff):
    return (d["stage"].isin(LATER_STAGE) | (d["amount_usd"].fillna(0) >= cutoff)).to_numpy()

cutoffs = [250_000, 500_000, 1_000_000, 2_000_000, 3_000_000, 5_000_000, 10_000_000, np.inf]
bl = pd.DataFrame([{"cutoff_usd": c, "cost_units": cost(train["exited"], baseline_rule(train, c)),
                    "meetings_per_100": 100 * baseline_rule(train, c).mean()} for c in cutoffs])
BASELINE_CUTOFF = float(bl.loc[bl["cost_units"].idxmin(), "cutoff_usd"])
print(bl.round(1).to_string(index=False))
print("\nchosen baseline: Series A or later, or first round >= $", f"{BASELINE_CUTOFF:,.0f}")

 cutoff_usd  cost_units  meetings_per_100
   250000.0        5599              85.5
   500000.0        5579              81.5
  1000000.0        5647              76.0
  2000000.0        5649              69.9
  3000000.0        5686              67.1
  5000000.0        5768              63.6
 10000000.0        6197              58.6
        inf        6624              52.6

chosen baseline: Series A or later, or first round >= $ 500,000


The two models. Preprocessing lives inside each pipeline, so imputation, encoding and scaling are learned from training rows only.

In [20]:
def make_logit(C=1.0, num_cols=None):
    num_cols = NUM if num_cols is None else num_cols
    num = Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", StandardScaler())])
    prep = ColumnTransformer([("num", num, num_cols),
                              ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False), CAT)])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(C=C, max_iter=5000))])

def make_gbm(learning_rate=0.05, max_iter=100, max_leaf_nodes=7, min_samples_leaf=100, l2_regularization=1.0, num_cols=None):
    num_cols = NUM if num_cols is None else num_cols
    prep = ColumnTransformer([("num", "passthrough", num_cols),
                              ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False), CAT)])
    clf = HistGradientBoostingClassifier(learning_rate=learning_rate, max_iter=max_iter, max_leaf_nodes=max_leaf_nodes,
                                         min_samples_leaf=min_samples_leaf, l2_regularization=l2_regularization,
                                         early_stopping=False, random_state=SEED)
    return Pipeline([("prep", prep), ("clf", clf)])
print("pipelines defined")

pipelines defined


Tune each model on the validation slice (fit on the earlier part of train), with a small grid, and keep the setting with the best validation AUC.

In [21]:
grid = [("logistic", make_logit, dict(C=C)) for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]] + [
    ("gbm", make_gbm, dict(learning_rate=0.05, max_iter=60, max_leaf_nodes=5, min_samples_leaf=100)),
    ("gbm", make_gbm, dict(learning_rate=0.03, max_iter=150, max_leaf_nodes=7, min_samples_leaf=80)),
    ("gbm", make_gbm, dict(learning_rate=0.05, max_iter=100, max_leaf_nodes=7, min_samples_leaf=100, l2_regularization=5.0)),
    ("gbm", make_gbm, dict(learning_rate=0.05, max_iter=200, max_leaf_nodes=15, min_samples_leaf=40)),
]
rows = []
for name, maker, params in grid:
    p = maker(**params).fit(fit_part[FEATURES], fit_part["exited"]).predict_proba(val[FEATURES])[:, 1]
    t, c = best_threshold(val["exited"].to_numpy(), p)
    rows.append(dict(model=name, params=params, val_auc=roc_auc_score(val["exited"], p), val_best_threshold=t, val_cost_units=c))
tuning = pd.DataFrame(rows)
print(tuning.round(4).to_string(index=False))

best = {m: tuning[tuning.model == m].sort_values("val_auc", ascending=False).iloc[0] for m in ["logistic", "gbm"]}
BEST_PARAMS = {m: dict(best[m]["params"]) for m in best}
THRESHOLD = {m: float(best[m]["val_best_threshold"]) for m in best}
print("\nchosen settings:", BEST_PARAMS)
print("thresholds chosen on validation by cost:", THRESHOLD)

   model                                                                                                           params  val_auc  val_best_threshold  val_cost_units
logistic                                                                                                     {'C': 0.001}   0.6856               0.135             949
logistic                                                                                                     {'C': 0.003}   0.6912               0.125             935
logistic                                                                                                      {'C': 0.01}   0.6937               0.130             933
logistic                                                                                                      {'C': 0.03}   0.6917               0.115             910
logistic                                                                                                       {'C': 0.1}   0.6862               0.110             92

Refit both chosen models on the full training set. The test set has not been touched yet.

In [22]:
models = {"logistic": make_logit(**BEST_PARAMS["logistic"]), "gbm": make_gbm(**BEST_PARAMS["gbm"])}
for m in models.values():
    m.fit(train[FEATURES], train["exited"])
p_test = {name: m.predict_proba(test[FEATURES])[:, 1] for name, m in models.items()}
print("fitted on", len(train), "training rows")

fitted on 6566 training rows


## Step 5 · Evaluate

The test-set scorecard: both baselines and both models, each model at the threshold it got on validation. Business unit: dollars of screening cost per 100 startups.

In [23]:
y_test = test["exited"].to_numpy()
b1_test = baseline_rule(test, BASELINE_CUTOFF)
card = pd.DataFrame([
    scorecard("predict no exit (meet none)", y_test, np.zeros(len(test), bool)),
    scorecard("meet everyone", y_test, np.ones(len(test), bool)),
    scorecard(f"baseline: Series A+ or >= ${BASELINE_CUTOFF/1e6:g}M", y_test, b1_test),
    scorecard(f"logistic @ {THRESHOLD['logistic']}", y_test, p_test["logistic"] >= THRESHOLD["logistic"], p_test["logistic"]),
    scorecard(f"gbm @ {THRESHOLD['gbm']}", y_test, p_test["gbm"] >= THRESHOLD["gbm"], p_test["gbm"]),
])
card["pr_auc"] = [np.nan, np.nan, np.nan, average_precision_score(y_test, p_test["logistic"]), average_precision_score(y_test, p_test["gbm"])]
print(card.round(3).to_string(index=False))

                           rule  meetings_per_100  exits_caught  exits_missed  recall  precision  accuracy  cost_units  cost_per_100_usd   auc  pr_auc
    predict no exit (meet none)             0.000             0           189   0.000      0.000     0.885        1890         34405.340   NaN     NaN
                  meet everyone           100.000           189             0   1.000      0.115     0.115        1459         26559.466   NaN     NaN
baseline: Series A+ or >= $0.5M            74.090           156            33   0.825      0.128     0.334        1395         25394.417   NaN     NaN
                logistic @ 0.13            51.335           142            47   0.751      0.168     0.544        1174         21371.359 0.680   0.202
                    gbm @ 0.115            64.927           160            29   0.847      0.150     0.430        1200         21844.660 0.677   0.192


The rule set before looking at test: ship the gradient-boosted model only if it beats logistic regression on test cost **and** test AUC; otherwise ship the simpler model.

In [24]:
lg, gb = card.iloc[3], card.iloc[4]
gbm_wins = (gb["cost_units"] < lg["cost_units"]) and (gb["auc"] > lg["auc"])
SHIP = "gbm" if gbm_wins else "logistic"
model, p, T = models[SHIP], p_test[SHIP], THRESHOLD[SHIP]
MAKER = make_logit if SHIP == "logistic" else make_gbm
print(f"logistic: cost {lg['cost_units']:.0f}, AUC {lg['auc']:.3f}  |  gbm: cost {gb['cost_units']:.0f}, AUC {gb['auc']:.3f}")
print("SHIP:", SHIP, "at threshold", T)

logistic: cost 1174, AUC 0.680  |  gbm: cost 1200, AUC 0.677
SHIP: logistic at threshold 0.13


Same meeting budget: if the screener took exactly as many meetings as the baseline rule does, how many exits would each catch?

In [25]:
k = int(b1_test.sum())
top_k = np.zeros(len(test), bool)
top_k[np.argsort(-p, kind="stable")[:k]] = True
same_budget = pd.DataFrame([
    dict(rule="baseline rule", meetings=k, exits_caught=confusion(y_test, b1_test)["tp"]),
    dict(rule=f"{SHIP}: top {k} by score", meetings=k, exits_caught=confusion(y_test, top_k)["tp"]),
])
same_budget["share_of_exits"] = same_budget["exits_caught"] / y_test.sum()
print(same_budget.round(3).to_string(index=False))

                       rule  meetings  exits_caught  share_of_exits
              baseline rule      1221           156           0.825
logistic: top 1221 by score      1221           167           0.884


Confusion matrix of the shipped model on test at the chosen threshold, and what the mistakes cost.

In [26]:
cm = confusion(y_test, p >= T)
print(pd.DataFrame([[cm["tp"], cm["fn"]], [cm["fp"], cm["tn"]]],
                   index=["actually exited", "did not exit"], columns=["meet (flagged)", "pass"]))
print(f"\nwasted meetings: {cm['fp']} x ${MEETING_COST_USD} = ${cm['fp']*MEETING_COST_USD:,}")
print(f"missed exits:    {cm['fn']} x ${MEETING_COST_USD*COST_FN} = ${cm['fn']*MEETING_COST_USD*COST_FN:,}")
print(f"total: ${(cm['fp']*COST_FP + cm['fn']*COST_FN)*MEETING_COST_USD:,} for {len(test):,} startups")

                 meet (flagged)  pass
actually exited             142    47
did not exit                704   755

wasted meetings: 704 x $300 = $211,200
missed exits:    47 x $3000 = $141,000
total: $352,200 for 1,648 startups


Decision 5's alternatives: what the default 0.5 cutoff and the textbook 1/11 cutoff would have done on test, next to the chosen one.

In [27]:
alts = pd.DataFrame([scorecard(f"threshold {t}", y_test, p >= t, p) for t in [0.5, round(COST_FP / (COST_FP + COST_FN), 3), T]])
alts["source"] = ["scikit-learn default (.predict)", "theory: 1/(1+10), assumes calibrated probabilities", "chosen on validation by cost"]
print(alts[["rule", "source", "meetings_per_100", "exits_caught", "cost_units", "cost_per_100_usd"]].round(1).to_string(index=False))
print("highest predicted probability on test:", round(p.max(), 3))

           rule                                             source  meetings_per_100  exits_caught  cost_units  cost_per_100_usd
  threshold 0.5                    scikit-learn default (.predict)               0.0             0        1890           34405.3
threshold 0.091 theory: 1/(1+10), assumes calibrated probabilities              78.9           171        1309           23828.9
 threshold 0.13                       chosen on validation by cost              51.3           142        1174           21371.4
highest predicted probability on test: 0.351


What would change the recommendation: re-solve the threshold on validation for other cost ratios and see where the model stops beating 'meet everyone' and the baseline on test.

In [28]:
p_val_ship = MAKER(**BEST_PARAMS[SHIP]).fit(fit_part[FEATURES], fit_part["exited"]).predict_proba(val[FEATURES])[:, 1]
sens = []
for ratio in [2, 3, 5, 7, 10, 15, 20, 30, 50]:
    t, _ = best_threshold(val["exited"].to_numpy(), p_val_ship, 1, ratio)
    sens.append(dict(fn_to_fp_ratio=ratio, threshold=t, meetings_per_100=100 * (p >= t).mean(),
                     model_cost=cost(y_test, p >= t, 1, ratio), baseline_cost=cost(y_test, b1_test, 1, ratio),
                     meet_all_cost=cost(y_test, np.ones(len(test), bool), 1, ratio),
                     meet_none_cost=cost(y_test, np.zeros(len(test), bool), 1, ratio)))
sens = pd.DataFrame(sens)
sens["model_beats_all_others"] = sens["model_cost"] < sens[["baseline_cost", "meet_all_cost", "meet_none_cost"]].min(axis=1)
print(sens.round(3).to_string(index=False))

 fn_to_fp_ratio  threshold  meetings_per_100  model_cost  baseline_cost  meet_all_cost  meet_none_cost  model_beats_all_others
              2      0.245             3.519         403           1131           1459             378                   False
              3      0.230             5.522         562           1164           1459             567                    True
              5      0.195            14.502         824           1230           1459             945                    True
              7      0.130            51.335        1033           1296           1459            1323                    True
             10      0.130            51.335        1174           1395           1459            1890                    True
             15      0.095            76.214        1371           1560           1459            2835                    True
             20      0.095            76.214        1466           1725           1459            3780         

Calibration check: is a predicted 15% roughly a real 15%? Compare predicted and actual exit rates by score decile on test.

In [29]:
dec = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)
calib = pd.DataFrame({"p": p, "y": y_test, "decile": dec}).groupby("decile").agg(n=("y", "size"), mean_predicted=("p", "mean"), actual_rate=("y", "mean"))
print(calib.round(3))
print("\nmean predicted on test:", round(p.mean(), 3), " actual test base rate:", round(y_test.mean(), 3))

          n  mean_predicted  actual_rate
decile                                  
0       165           0.052        0.055
1       165           0.080        0.048
2       165           0.096        0.048
3       164           0.110        0.079
4       165           0.125        0.067
5       165           0.140        0.091
6       164           0.155        0.152
7       165           0.171        0.152
8       165           0.194        0.206
9       165           0.241        0.248

mean predicted on test: 0.136  actual test base rate: 0.115


Where it is weakest: AUC and recall at the threshold by segment, on test, for segments with at least 15 exits.

In [30]:
seg_defs = {
    "stage": test["stage"],
    "region": np.where(test["country_code"] == "USA", "US", "non-US"),
    "investors recorded": np.where(test["n_investors"].notna(), "recorded", "not recorded"),
    "amount recorded": np.where(test["amount_usd"].notna(), "recorded", "not recorded"),
    "category": np.where(test["category"].isin(["software", "web", "biotech", "mobile", "enterprise", "games_video"]), test["category"], "all other"),
}
seg_rows = []
for seg, labels in seg_defs.items():
    for lab in pd.unique(labels):
        m = np.asarray(labels) == lab
        if y_test[m].sum() >= 15 and (1 - y_test[m]).sum() >= 15:
            c = confusion(y_test[m], p[m] >= T)
            seg_rows.append(dict(segment=seg, value=lab, n=int(m.sum()), exits=int(y_test[m].sum()),
                                 auc=roc_auc_score(y_test[m], p[m]), recall=c["tp"] / (c["tp"] + c["fn"]),
                                 meetings_per_100=100 * (p[m] >= T).mean()))
segments = pd.DataFrame(seg_rows).sort_values("auc")
print(segments.round(3).to_string(index=False))
WEAK = segments.iloc[0]
print("\nweakest segment:", WEAK["segment"], "=", WEAK["value"], "AUC", round(WEAK["auc"], 3), "vs overall", round(roc_auc_score(y_test, p), 3))

           segment             value    n  exits   auc  recall  meetings_per_100
investors recorded      not recorded  316     20 0.533   0.000             0.000
             stage series-b-or-later  212     35 0.599   0.914            83.491
            region            non-US  689     45 0.609   0.422            24.819
            region                US  959    144 0.646   0.854            70.386
          category         all other  771     91 0.655   0.714            49.805
             stage venture-unlabeled  395     24 0.663   0.750            48.101
   amount recorded      not recorded  220     29 0.679   0.655            42.273
investors recorded          recorded 1332    169 0.680   0.840            63.514
   amount recorded          recorded 1428    160 0.682   0.769            52.731
             stage        seed-angel  491     49 0.685   0.510            24.644
             stage          series-a  518     77 0.704   0.844            66.216
          category          

Decision 4's alternative: what a random split would have reported instead of the time split (same model settings, same seed).

In [31]:
rtr, rte = train_test_split(co, test_size=len(test), random_state=SEED, stratify=co["exited"])
p_rand = MAKER(**BEST_PARAMS[SHIP]).fit(rtr[FEATURES], rtr["exited"]).predict_proba(rte[FEATURES])[:, 1]
split_compare = pd.DataFrame([
    dict(split="time (used)", test_rows=len(test), test_base_rate=y_test.mean(), test_auc=roc_auc_score(y_test, p), mean_predicted=p.mean(),
         test_cohorts=f"{test.first_date.min().date()} to {test.first_date.max().date()}"),
    dict(split="random", test_rows=len(rte), test_base_rate=rte["exited"].mean(), test_auc=roc_auc_score(rte["exited"], p_rand), mean_predicted=p_rand.mean(),
         test_cohorts=f"{rte.first_date.min().date()} to {rte.first_date.max().date()}"),
])
print(split_compare.round(3).to_string(index=False))

      split  test_rows  test_base_rate  test_auc  mean_predicted             test_cohorts
time (used)       1648           0.115      0.68           0.136 2008-02-25 to 2008-12-12
     random       1648           0.144      0.67           0.142 2000-01-01 to 2008-12-10


Leakage check on the most suspicious kept feature, `n_investors`: refit without it, then test whether its signal survives on rounds recorded within a year (before an exit could be known).

In [32]:
FEAT_NO_INV = [f for f in FEATURES if f != "n_investors"]
m_no_inv = MAKER(**BEST_PARAMS[SHIP], num_cols=[f for f in NUM if f != "n_investors"]).fit(train[FEAT_NO_INV], train["exited"])
p_no_inv = m_no_inv.predict_proba(test[FEAT_NO_INV])[:, 1]

lag_days = (test["record_created"] - test["first_date"]).dt.days.to_numpy()
prompt = lag_days <= 365
leak = []
for name, mask in [("all test rows", np.ones(len(test), bool)), ("recorded within 1 year of the round", prompt), ("recorded 1+ years later (backfilled)", ~prompt)]:
    leak.append(dict(rows=name, n=int(mask.sum()), exits=int(y_test[mask].sum()),
                     auc_with=roc_auc_score(y_test[mask], p[mask]), auc_without=roc_auc_score(y_test[mask], p_no_inv[mask])))
leak = pd.DataFrame(leak)
leak["drop"] = leak["auc_with"] - leak["auc_without"]
print(leak.round(3).to_string(index=False))

lag_all = (co["record_created"] - co["first_date"]).dt.days
rate = co.groupby([np.where(lag_all <= 365, "prompt", "backfilled"), co["n_investors"].isna().map({True: "investors missing", False: "investors recorded"})])["exited"].agg(["size", "mean"])
print("\nexit rate by recording lag and investor-count missingness (all rows):")
print(rate.round(3))

                                rows    n  exits  auc_with  auc_without  drop
                       all test rows 1648    189     0.680        0.647 0.033
 recorded within 1 year of the round 1045    137     0.690        0.647 0.043
recorded 1+ years later (backfilled)  603     52     0.641        0.623 0.018



exit rate by recording lag and investor-count missingness (all rows):
                               size   mean
           n_investors                    
backfilled investors missing    964  0.063
           investors recorded  5446  0.157
prompt     investors missing    367  0.060
           investors recorded  1437  0.173


What the excluded leaky columns would have done: add each one to the shipped model and refit. These are the columns an agent tends to include because they raise the score.

In [33]:
leaky_rows = []
for col in ["funding_rounds", "funding_total_usd", "status"]:
    tr_l, te_l = train.copy(), test.copy()
    if col == "status":
        tr_l["status_exited"] = tr_l["status"].isin(["acquired", "ipo"]).astype(float)
        te_l["status_exited"] = te_l["status"].isin(["acquired", "ipo"]).astype(float)
        col = "status_exited"
    m_l = MAKER(**BEST_PARAMS[SHIP], num_cols=NUM + [col]).fit(tr_l[FEATURES + [col]], tr_l["exited"])
    leaky_rows.append(dict(added_column=col, test_auc=roc_auc_score(y_test, m_l.predict_proba(te_l[FEATURES + [col]])[:, 1])))
print("shipped model test AUC:", round(roc_auc_score(y_test, p), 3))
print(pd.DataFrame(leaky_rows).round(3).to_string(index=False))

shipped model test AUC: 0.68
     added_column  test_auc
   funding_rounds     0.676
funding_total_usd     0.681
    status_exited     0.997


Traced number 1 (for verification.md): rebuild the test set's exit count and base rate straight from the CSVs by a different route (sort + drop_duplicates and merges, not the groupby pipeline above or the app).

In [34]:
raw_r = pd.read_csv(DATA / "funding_rounds.csv", parse_dates=["funded_at"])
raw_c = pd.read_csv(DATA / "companies.csv", low_memory=False)
raw_a = pd.read_csv(DATA / "acquisitions.csv", parse_dates=["acquired_at"])
raw_i = pd.read_csv(DATA / "ipos.csv", parse_dates=["public_at"])
raw_r = raw_r[raw_r["object_id"].isin(raw_c["id"])]
firsts = raw_r.dropna(subset=["funded_at"]).sort_values("funded_at").drop_duplicates("object_id")[["object_id", "funded_at"]]
ex = pd.concat([raw_a.rename(columns={"acquired_object_id": "object_id", "acquired_at": "d"})[["object_id", "d"]],
                raw_i.rename(columns={"public_at": "d"})[["object_id", "d"]]]).dropna().sort_values("d").drop_duplicates("object_id")
t = firsts.merge(ex, on="object_id", how="left").merge(raw_c[["id", "status"]], left_on="object_id", right_on="id")
undated = set(raw_a.loc[raw_a["acquired_at"].isna(), "acquired_object_id"]) | set(raw_i.loc[raw_i["public_at"].isna(), "object_id"])
t = t[~(t["d"] < t["funded_at"])]
t = t[(t["funded_at"] >= TEST_START) & (t["funded_at"] <= "2008-12-12")]
t = t[~(t["d"].isna() & (t["status"].isin(["acquired", "ipo"]) | t["object_id"].isin(undated)))]
t_exit = int((t["d"] <= t["funded_at"] + pd.DateOffset(years=5)).sum())   # NaT compares False
print(f"independent route: {len(t)} test startups, {t_exit} exited, base rate {t_exit/len(t):.1%}")
print(f"pipeline:          {len(test)} test startups, {y_test.sum()} exited, base rate {y_test.mean():.1%}")

independent route: 1648 test startups, 189 exited, base rate 11.5%
pipeline:          1648 test startups, 189 exited, base rate 11.5%


Traced number 2 (for verification.md): the app's example startup. Recompute its probability by hand from the logistic regression's intercept and coefficients, without calling `predict_proba`.

In [35]:
EXAMPLE = pd.DataFrame([dict(amount_usd=5_000_000, n_investors=3, company_age_years=2.0, stage="series-a",
                             category="software", country="USA", us_state="CA")])
p_pipeline = float(model.predict_proba(EXAMPLE[FEATURES])[:, 1][0])
if SHIP == "logistic":
    prep, clf = model.named_steps["prep"], model.named_steps["clf"]
    numpipe = prep.named_transformers_["num"]
    x_num = np.log1p(EXAMPLE[NUM].to_numpy(dtype=float))[0]
    med = numpipe.named_steps["impute"].statistics_
    x_num = np.where(np.isnan(x_num), med, x_num)
    ind = np.isnan(np.log1p(EXAMPLE[NUM].to_numpy(dtype=float))[0])[numpipe.named_steps["impute"].indicator_.features_].astype(float)
    sc = numpipe.named_steps["scale"]
    z = (np.concatenate([x_num, ind]) - sc.mean_) / sc.scale_
    ohe = prep.named_transformers_["cat"]
    onehot = []
    for j, col in enumerate(CAT):
        cats = list(ohe.categories_[j])
        infreq = ohe.infrequent_categories_[j]
        kept = [c for c in cats if infreq is None or c not in set(infreq)]
        v = EXAMPLE[col].iloc[0]
        vec = [1.0 if v == c else 0.0 for c in kept]
        if infreq is not None and len(infreq):
            vec.append(1.0 if v in set(infreq) else 0.0)
        onehot += vec
    xfull = np.concatenate([z, onehot])
    logit = clf.intercept_[0] + float(np.dot(clf.coef_[0], xfull))
    p_hand = 1 / (1 + np.exp(-logit))
    print(f"by hand: logit = {logit:.4f}, p = {p_hand:.4f}")
print(f"pipeline predict_proba: p = {p_pipeline:.4f}")

by hand: logit = -1.2753, p = 0.2184
pipeline predict_proba: p = 0.2184


Which inputs move the score: the shipped model's largest coefficients (logistic) on the standardized scale.

In [36]:
if SHIP == "logistic":
    names = model.named_steps["prep"].get_feature_names_out()
    coefs = pd.Series(model.named_steps["clf"].coef_[0], index=names).sort_values()
    print(coefs.round(3).to_string())

num__missingindicator_n_investors         -0.275
cat__us_state_non-US                      -0.190
cat__category_biotech                     -0.172
cat__category_hardware                    -0.148
cat__us_state_US-other                    -0.125
cat__country_FRA                          -0.116
cat__category_infrequent_sklearn          -0.110
cat__country_GBR                          -0.093
cat__category_medical                     -0.088
cat__category_cleantech                   -0.082
cat__category_analytics                   -0.070
cat__stage_seed-angel                     -0.069
cat__country_other                        -0.069
cat__country_DEU                          -0.068
cat__category_public_relations            -0.057
cat__country_IND                          -0.053
cat__category_unknown                     -0.051
cat__stage_venture-unlabeled              -0.032
cat__category_semiconductor               -0.029
cat__stage_infrequent_sklearn             -0.027
cat__category_consul

Save what the app needs: the fitted pipeline (`model.joblib`), the scored test set for the 'how good is this' view, and a metadata file with the threshold, costs and input ranges.

In [37]:
Path("app_data").mkdir(exist_ok=True)
joblib.dump(model, "model.joblib")

scored = test[["first_date", "stage", "category", "country", "us_state", "amount_usd", "n_investors", "company_age_years", "exited"]].copy()
scored["p_exit"] = p
scored["baseline_flag"] = b1_test.astype(int)
scored["region"] = np.where(test["country_code"] == "USA", "US", "non-US")
scored["first_date"] = scored["first_date"].dt.strftime("%Y-%m-%d")
scored.round(6).to_csv("app_data/test_scored.csv", index=False)

meta = dict(
    shipped_model=SHIP, shipped_params=BEST_PARAMS[SHIP], threshold=T,
    cost_fp=COST_FP, cost_fn=COST_FN, meeting_cost_usd=MEETING_COST_USD,
    baseline_cutoff_usd=BASELINE_CUTOFF, baseline_stages=LATER_STAGE,
    train_rows=len(train), test_rows=len(test), train_base_rate=float(train["exited"].mean()), test_base_rate=float(y_test.mean()),
    train_dates=[str(train.first_date.min().date()), str(train.first_date.max().date())],
    test_dates=[str(test.first_date.min().date()), str(test.first_date.max().date())],
    test_auc=float(roc_auc_score(y_test, p)), test_auc_other=float(roc_auc_score(y_test, p_test["gbm" if SHIP == "logistic" else "logistic"])),
    ranges=dict(amount_usd=[float(train.amount_usd.quantile(0.01)), float(train.amount_usd.quantile(0.99))],
                n_investors=[int(train.n_investors.min()), int(train.n_investors.max())],
                company_age_years=[0.0, float(train.company_age_years.quantile(0.99))]),
    categories={c: train[c].value_counts().to_dict() for c in CAT},
    weakest_segment=dict(segment=WEAK["segment"], value=WEAK["value"], auc=float(WEAK["auc"]), recall=float(WEAK["recall"]), exits=int(WEAK["exits"]), n=int(WEAK["n"])),
    segments=segments.round(4).to_dict(orient="records"),
    random_split_auc=float(split_compare.iloc[1]["test_auc"]),
    leakage_check=leak.round(4).to_dict(orient="records"),
    sensitivity=sens.round(4).to_dict(orient="records"),
    example=EXAMPLE.iloc[0].to_dict(), example_p=p_pipeline,
)
Path("app_data/meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("saved model.joblib, app_data/test_scored.csv, app_data/meta.json")
print(json.dumps({k: meta[k] for k in ["shipped_model", "threshold", "test_auc", "test_base_rate", "baseline_cutoff_usd"]}, indent=1))

saved model.joblib, app_data/test_scored.csv, app_data/meta.json
{
 "shipped_model": "logistic",
 "threshold": 0.13,
 "test_auc": 0.6801389659511661,
 "test_base_rate": 0.11468446601941748,
 "baseline_cutoff_usd": 500000.0
}
